# Chapter 3 — Coding attention mechanisms

This notebook develops attention from its simplest ingredients: token embeddings, dot-product similarity, and normalized attention
weights. Each step keeps the tensors small enough to inspect by hand before moving toward trainable self-attention.

## Learning goals

By the end of this section, you should be able to:

- interpret a token embedding matrix;
- compute attention scores with dot products;
- distinguish raw scores from normalized attention weights; and
- explain why softmax is preferred for attention normalization.

## 3.1 Representing the input sequence

The sentence “Your journey starts with one step” contains six tokens. Each token is represented by a three-dimensional embedding, so
`inputs` has shape `(6, 3)`:

- rows correspond to token positions;
- columns correspond to embedding features.

These fixed vectors let us focus on attention calculations before introducing learned query, key, and value projections.

In [2]:
import torch

# Each row is a three-dimensional embedding for one token in the sentence.
inputs = torch.tensor(
    [
        [0.43, 0.15, 0.89],  # Your     (x^1)
        [0.55, 0.87, 0.66],  # journey  (x^2)
        [0.57, 0.85, 0.64],  # starts   (x^3)
        [0.22, 0.58, 0.33],  # with     (x^4)
        [0.77, 0.25, 0.10],  # one      (x^5)
        [0.05, 0.80, 0.55],  # step     (x^6)
    ]
)

C:\Users\giloz\dev\build-llms-from-scratch-companion\.venv\Lib\site-packages\torch\_subclasses\functional_tensor.py:368: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


## 3.2 Computing attention scores for one query

To determine which tokens are relevant to “journey,” use its embedding as the **query**. Compute one dot product between that query
and every input embedding.

The result `attn_scores_2` has shape `(6,)`: one raw alignment score for each token. A larger score means stronger vector alignment,
but these scores are not yet probabilities and do not need to sum to one.

In [5]:
# Use the second token, "journey," as the query for this example.
query = inputs[1]
print(query)

# torch.empty allocates storage without initializing values; every slot is
# overwritten in the loop before the scores are used.
attn_scores_2 = torch.empty(inputs.shape[0])
print(attn_scores_2.shape)

# A dot product measures the alignment between the query and each input token.
for i, x_i in enumerate(inputs):
    attn_scores_2[i] = torch.dot(x_i, query)
print(attn_scores_2)

tensor([0.5500, 0.8700, 0.6600])
torch.Size([6])
tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


### Dot product worked example

For vectors $a$ and $b$, their dot product multiplies corresponding components and sums the results:

$$
a \cdot b = \sum_j a_j b_j
$$

The next cell computes the score between the first token and the query manually, then compares it with `torch.dot`. This confirms
what the vectorized PyTorch operation does.

In [7]:
# Expand one dot product into scalar multiplications and additions.
res = 0.0
for idx, _ in enumerate(inputs[0]):
    res += inputs[0][idx] * query[idx]

# Both calculations should produce the same attention score.
print(res)
print(torch.dot(inputs[0], query))

tensor(0.9544)
tensor(0.9544)


## 3.3 Normalizing attention scores

Attention weights express each token's relative contribution. A simple first approach divides every score by the sum of all scores.
This produces weights that sum to one for this positive-valued example.

This normalization is useful for intuition, but it is not the standard attention rule. It can behave poorly when scores are negative
or their sum is near zero.

In [8]:
# Divide each score by the total to obtain nonnegative weights summing to one.
attn_weights_2_tmp = attn_scores_2 / attn_scores_2.sum()
print("Attention weights:", attn_weights_2_tmp)
print("Sum:", attn_weights_2_tmp.sum())

Attention weights: tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])
Sum: tensor(1.0000)


### Normalizing with softmax

Softmax exponentiates each score and divides by the sum of all exponentiated scores:

$$
\mathrm{softmax}(s)_i =
\frac{\exp(s_i)}{\displaystyle\sum_{j=1}^{n} \exp(s_j)}
$$

Here, $s_i$ is the score for token $i$, and $n$ is the number of tokens. The output is positive, sums to one, and gives relatively
larger weights to stronger scores. The implementation below is educational; production code should use `torch.softmax`, which is
optimized and numerically more stable.

In [9]:
def softmax_naive(x: torch.Tensor) -> torch.Tensor:
    """Convert a one-dimensional score tensor into normalized probabilities."""
    return torch.exp(x) / torch.exp(x).sum(dim=0)


# Exponentiation emphasizes larger alignment scores before normalization.
attn_weights_2_naive = softmax_naive(attn_scores_2)
print("Attention weights:", attn_weights_2_naive)
print("Sum:", attn_weights_2_naive.sum())

Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)


### Using PyTorch's stable softmax

The naive implementation repeats `torch.exp(x)` and can overflow for large scores. `torch.softmax` performs the same normalization
with a more numerically stable and optimized implementation.

Here, `dim=0` means “normalize across the token dimension.” Because `attn_scores_2` contains one score per token, the resulting six
weights describe how strongly this query attends to each position.

In [10]:
# Normalize across the six token scores with PyTorch's stable implementation.
attn_weights_2 = torch.softmax(attn_scores_2, dim=0)
print("Attention weights:", attn_weights_2)
print("Sum:", attn_weights_2.sum())

Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)


## 3.4 Computing the context vector

Attention weights become useful when they are applied to the input embeddings. Multiply each input vector by its scalar attention
weight and add the weighted vectors together.

In compact notation, the context vector for query 2 is:

$$
\mathbf{z}^{(2)} = \sum_i \alpha_i^{(2)} \mathbf{x}^{(i)}
$$

Each $\alpha_i^{(2)}$ is a scalar weight and each $\mathbf{x}^{(i)}$ is a three-dimensional input vector. Their weighted sum is
therefore another three-dimensional vector. Tokens receiving larger weights contribute more strongly to the result.

In [11]:
# Reuse "journey" as the query whose context vector we are constructing.
query = inputs[1]
# Start with a zero vector having the same embedding dimensions as the query.
context_vec_2 = torch.zeros(query.shape)

# Scale each input vector by its attention weight, then add the results.
for i, x_i in enumerate(inputs):
    context_vec_2 += attn_weights_2[i] * x_i

print(context_vec_2)

tensor([0.4419, 0.6515, 0.5683])


## Checkpoint

For the query token “journey,” the attention workflow is now complete:

1. select the query embedding;
2. compute a dot-product score against every input token;
3. apply `torch.softmax` to obtain positive weights that sum to one; and
4. calculate a weighted sum of all input embeddings to produce the query's context vector.

The context vector has the same width as an input embedding, but it contains information gathered from the entire sequence. The next
step is to generalize this computation from one query to every token.